# A few labels and many predictions: prediction-powered inference

Some outcomes are expensive to measure and cheap to predict. Linking survey respondents to
administrative records, coding documents by hand, or running a lab test costs money per unit. A
model trained elsewhere (a machine-learning model fitted on another data set, an LLM) can predict
the outcome for everyone. Analysing the predictions as if they were data is biased whenever the
model is wrong in a systematic way. Using only the labelled units throws away the information in
the predictions.

**Prediction-powered inference** (Angelopoulos, Bates, Fannjiang, Jordan and Zrnic, 2023) combines
both. It estimates the parameter from the predictions and corrects it with the prediction errors on
the labelled units. For a mean,

$$
\hat\theta_\lambda = \lambda\,\frac1N\sum_{j=1}^N \hat y^{u}_j
  + \frac1n\sum_{i=1}^n \bigl(y_i - \lambda \hat y_i\bigr),
$$

which is unbiased however good or bad the predictions are. PPI++ (Angelopoulos, Duchi and Zrnic,
2023) estimates the weight $\lambda$ that minimises the variance. The better the predictions, the
more the unlabelled units count.

We use the 401(k) data of DoubleML (9,915 households). Suppose that 401(k) **participation** could
only be measured by linking records, which we can afford for 500 households, while a model trained
on another survey predicts participation for every household.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import dml_rs

pd.set_option("display.precision", 4)

df = dml_rs.datasets.fetch_401K(return_type="DataFrame")
features = [
    "age",
    "inc",
    "fsize",
    "educ",
    "db",
    "marr",
    "twoearn",
    "e401",
    "pira",
    "hown",
]
F = df[features].to_numpy(float)
y = df["p401"].to_numpy(float)

rng = np.random.default_rng(0)
order = rng.permutation(len(df))
other_survey, study = order[:3000], order[3000:]

# The "external" model: trained on another survey that only covered
# households with below-median income, so it does not fit richer ones.
low_income = F[other_survey, features.index("inc")] < np.median(df["inc"])
train = other_survey[low_income]
external = dml_rs.HistGradientBoostingClassifier(random_state=0)
external.fit(F[train], y[train])
prediction = external.predict_proba(F)[:, 1]

labelled, unlabelled = study[:500], study[500:]
truth = y[study].mean()
print(
    f"{len(labelled)} labelled, {len(unlabelled)} unlabelled households"
)
print(
    f"participation rate in the study (unknown in practice): {truth:.4f}"
)

500 labelled, 6415 unlabelled households
participation rate in the study (unknown in practice): 0.2620


## Three ways to estimate the participation rate

- **Classical:** the mean over the 500 labelled households. Unbiased, but noisy.
- **Imputation:** the mean prediction over all households. Very precise, but biased if the model
  is.
- **PPI++:** `dml_rs.PPI`, with the labelled outcomes, their predictions and the predictions for
  the unlabelled households.

In [2]:
ppi = dml_rs.PPI().fit(
    y[labelled], prediction[labelled], prediction[unlabelled]
)
imputed = prediction[study]
rows = {
    "classical": (ppi.classical_.coef_[0], ppi.classical_.se_[0]),
    "imputation": (imputed.mean(), imputed.std() / np.sqrt(len(imputed))),
    "PPI++": (ppi.coef_[0], ppi.se_[0]),
}
table = pd.DataFrame(rows, index=["estimate", "std err"]).T
table["95% CI"] = [
    f"[{e - 1.96 * s:.3f}, {e + 1.96 * s:.3f}]"
    for e, s in zip(table["estimate"], table["std err"])
]
table

,estimate,std err,95% CI
classical,0.2620,0.0197,"[0.223, 0.301]"
imputation,0.2173,0.0038,"[0.210, 0.225]"
PPI++,0.2573,0.0144,"[0.229, 0.286]"


In [3]:
print(f"lambda = {ppi.lam_:.3f}")
print(
    f"effective sample size: {ppi.effective_n_[0]:.0f} labelled households"
)

lambda = 0.934
effective sample size: 930 labelled households


The external model under-predicts participation for richer households, so the imputation is far
off, and its tiny interval, which reflects only the noise in the predictions and not their bias,
excludes the true rate. PPI++ is corrected by the 500 labels: it is close to the true rate and as
precise as the classical mean would be with almost twice as many labels.

## Regression coefficients

The same correction works for regression coefficients. How does participation depend on income
(in $10,000), age (in decades) and education? `estimand="logistic"` fits a logistic regression with
an intercept:

In [4]:
X = np.column_stack([df["inc"] / 10_000, df["age"] / 10, df["educ"]])
names = ["income", "age", "education"]
X = pd.DataFrame(X, columns=names)
logit = dml_rs.PPI("logistic").fit(
    y[labelled],
    prediction[labelled],
    prediction[unlabelled],
    X=X.iloc[labelled],
    X_unlabeled=X.iloc[unlabelled],
)
pd.DataFrame(
    {
        "PPI++": logit.coef_,
        "std err": logit.se_,
        "classical": logit.classical_.coef_,
        "classical std err": logit.classical_.se_,
        "effective n": logit.effective_n_,
    },
    index=logit.treatment_names_,
)

,PPI++,std err,classical,classical std err,effective n
intercept,-2.3727,0.5485,-2.8604,0.7882,1032.5864
income,0.2432,0.0351,0.2525,0.0549,1224.6862
age,0.0020,0.0764,0.0426,0.1007,868.0510
education,0.0252,0.0365,0.0499,0.0453,768.8985


## Repeated sampling

Does it hold up? Draw 300 different sets of 500 labelled households from the study and compare the
three estimates of the participation rate with the study's true rate, and the classical and PPI++
estimates of the income coefficient with its value on all study households:

In [5]:
full = dml_rs.PPI("logistic", lam=0.0).fit(
    y[study],
    y[study],
    y[study],
    X=X.iloc[study],
    X_unlabeled=X.iloc[study],
)
income_truth = full.coef_[1]

draws = []
for r in range(300):
    pick = np.random.default_rng(r).permutation(study)
    lab, unl = pick[:500], pick[500:]
    mean = dml_rs.PPI().fit(y[lab], prediction[lab], prediction[unl])
    reg = dml_rs.PPI("logistic").fit(
        y[lab],
        prediction[lab],
        prediction[unl],
        X=X.iloc[lab],
        X_unlabeled=X.iloc[unl],
    )
    for name, fit, j, target in [
        ("rate: classical", mean.classical_, 0, truth),
        ("rate: PPI++", mean, 0, truth),
        ("income: classical", reg.classical_, 1, income_truth),
        ("income: PPI++", reg, 1, income_truth),
    ]:
        lo, hi = fit.confint(0.95)[j]
        draws.append((name, lo <= target <= hi, hi - lo))
    e, s = (
        prediction[pick].mean(),
        prediction[pick].std() / np.sqrt(len(pick)),
    )
    draws.append(
        (
            "rate: imputation",
            e - 1.96 * s <= truth <= e + 1.96 * s,
            3.92 * s,
        )
    )

pd.DataFrame(draws, columns=["estimate", "covered", "width"]).groupby(
    "estimate", sort=False
).agg(coverage=("covered", "mean"), mean_width=("width", "mean"))

,coverage,mean_width
estimate,,
rate: classical,0.9567,0.0772
rate: PPI++,0.9633,0.0576
income: classical,0.9767,0.2006
income: PPI++,0.9667,0.1549
rate: imputation,0.0000,0.0148


The classical and PPI++ intervals cover at least 95% of the time (a little more, because the
labelled households are drawn from the study itself, which makes the intervals slightly
conservative), and the PPI++ intervals are about a quarter narrower. The imputation uses no labels, so
it is the same in every draw, and its interval misses the true rate every time.

## No external model? Cross-fit one

Without a model trained elsewhere, `learner=` fits one on the labelled households by
cross-fitting (cross-prediction-powered inference, Zrnic and Candès, 2024). Each labelled household
is predicted by a model that did not see it, and each unlabelled household gets the average of the
fold models' predictions:

In [6]:
cross = dml_rs.PPI(
    learner=dml_rs.HistGradientBoostingClassifier(random_state=0)
).fit(y[labelled], features=F[labelled], features_unlabeled=F[unlabelled])
print(
    f"estimate {cross.coef_[0]:.4f}, std err {cross.se_[0]:.4f} "
    f"(classical {cross.classical_.se_[0]:.4f}), "
    f"effective n {cross.effective_n_[0]:.0f}"
)

estimate 0.2746, std err 0.0144 (classical 0.0197), effective n 934


Here the model learned from 500 labels does about as well as the external one. Either way the
predictions can only cost precision, never validity: poor predictions get a small weight.

## Summary

- `dml_rs.PPI` estimates means (`"mean"`), linear (`"ols"`) and logistic (`"logistic"`) regression
  coefficients from labelled outcomes and predictions for labelled and unlabelled units.
- The default PPI++ weight `lam_` adapts to the quality of the predictions; `effective_n_` says how
  many labels the classical estimate would need for the same precision, and `classical_` is the
  labelled-only fit.
- The labelled units must be a random sample of all units. The results are ordinary fitted-model
  objects (`summary`, `confint`, `framework`, `confidence_sequence`).